<a href="https://colab.research.google.com/github/PouryaDadlouie98/Deep-Learning-mathematical-approach-/blob/main/Basis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning Foundations: From Autograd to PyTorch Tensors

Welcome to the foundational notebook of this Deep Learning journey. The goal is to build an intuitive, practical understanding of modern deep learning frameworks?bridging low-level math with production-ready PyTorch code.

---

### Roadmap of this Notebook
1. **Scalar Autograd Engine (From Scratch)**: Computational graphs, scalar differentiation, and manual gradient backpropagation.
2. **Modern PyTorch Autograd**: High-performance automatic differentiation using `torch.Tensor` and dynamic computational graphs.
3. **Tensor Fundamentals**: Creation, data types (`dtype`), shapes, and device management (`CPU` vs. `CUDA`).
4. **Tensor Indexing, Slicing & Reshaping**: Multi-dimensional slicing, batch indexing, and image formatting `(B, C, H, W)`.
5. **Optimization & nn.Module**: Building modular neural network layers and training loops.


# 1. Scalar Autograd Engine from Scratch
Exploring automatic differentiation at the scalar level: building a computational graph and backpropagating gradients manually.


In [ ]:
class Value:
  def __init__(self,data,_children=(),_op=''):
    self.data=data
    self.grad=0.0
    self._prev=set(_children)
    self._op=_op
    self._backward=lambda: None

  def __add__(self,other):
    out=Value(self.data+other.data,(self,other),'+')

    def _backward():
      self.grad+= 1.0*out.grad
      other.grad += 1.0*out.grad
    out._backward=_backward
    return out
  def __repr__(self):
     return f"Value(data={self.data}, grad={self.grad})"



In [ ]:

t=Value(0.0)
res=[]

for i in range(1,7):
  t=t+Value(float(i))
  res.append(t)
print(t)

print(res)
print(t._prev)


Value(data=21.0, grad=0.0)
[Value(data=1.0, grad=0.0), Value(data=3.0, grad=0.0), Value(data=6.0, grad=0.0), Value(data=10.0, grad=0.0), Value(data=15.0, grad=0.0), Value(data=21.0, grad=0.0)]
{Value(data=6.0, grad=0.0), Value(data=15.0, grad=0.0)}


In [ ]:
res[3].grad = 1.0
res[3]._backward()
print(res[3]._prev)

{Value(data=4.0, grad=1.0), Value(data=6.0, grad=1.0)}


# 2. PyTorch Autograd & Tensors
Moving from scalar micrograd to tensorized automatic differentiation with PyTorch.


In [ ]:
import torch

In [ ]:
x=torch.tensor([3,4],dtype=torch.float64,requires_grad=True)
y=torch.tensor([3,3],dtype=torch.float64,requires_grad=True)
z=torch.tensor([3,4],dtype=torch.float64,requires_grad=True)


In [ ]:
f=(x*y+y)*z
f.sum().backward()
print(x.grad,y.grad,z.grad)

tensor([ 9., 12.], dtype=torch.float64) tensor([12., 20.], dtype=torch.float64) tensor([12., 15.], dtype=torch.float64)


> <small>**Key Takeaway (`w.grad` & `b.grad`):**<br>
> ? **`w.grad`**: Sensitivity to feature scaling (slope).<br>
> ? **`b.grad`**: Sensitivity to baseline shift (threshold when $x=0$).<br>
> ? **Sign**: Negative $	o$ increase parameter; Positive $	o$ decrease parameter.<br>
> ? **Update Rule**: $	heta \leftarrow 	heta - \eta \cdot 	heta.	ext{grad}$ (slides down the error surface).</small>


## 2.1 Tensor Creation & Basic Attributes
Tensors are n-dimensional arrays optimized for GPU acceleration and automatic differentiation.
Key attributes to always inspect:
- `shape`: Size along each dimension
- `dtype`: Data type (e.g., `float32`, `int64`)
- `device`: Storage device (`cpu` or `cuda`)


In [ ]:

a = torch.tensor([1, 2, 3])
b = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
c = torch.zeros(4, 3)
d = torch.rand(2, 3, 4)

for t in [a, b, c, d]:
    print(t.shape, t.dtype, t.ndim)

torch.Size([3]) torch.int64 1
torch.Size([2, 3]) torch.float32 2
torch.Size([4, 3]) torch.float32 2
torch.Size([2, 3, 4]) torch.float32 3


## 2.2 Tensor Indexing & Slicing
Indexing in PyTorch is identical to NumPy slicing, but fundamental to preparing data for neural networks:
- **Batch indexing**: Accessing specific samples `batch[i]`
- **Channel extraction**: Selecting color channels `image[channel_idx, :, :]`
- **Spatial cropping**: Extracting height/width regions `image[:, y1:y2, x1:x2]`
- **Ellipsis (`...`)**: Shorthand to keep all preceding dimensions `tensor[..., 0]`
- **Boolean Masking**: Filtering values conditionally `tensor[tensor > threshold]`


In [ ]:
# 1. Standard 1D & 2D Indexing
x = torch.tensor([[10, 20, 30], [40, 50, 60], [70, 80, 90]])
print("Full 2D tensor:
", x)
print("First row:", x[0])
print("First column:", x[:, 0])
print("Last element (bottom-right):", x[-1, -1])
print("Sub-matrix (top-left 2x2):
", x[0:2, 0:2])

print("-" * 40)

# 2. Vision Indexing: 4D Image Batch (Batch, Channels, Height, Width)
# Imagine 4 RGB images of size 28x28
images = torch.rand(4, 3, 28, 28)
print("Batch shape:", images.shape)

# Grab the first image in the batch -> shape (3, 28, 28)
first_img = images[0]
print("First image shape:", first_img.shape)

# Extract only the Red channel (channel 0) across all images -> shape (4, 28, 28)
red_channels = images[:, 0, :, :]
print("All Red channels shape:", red_channels.shape)

# Crop a 10x10 patch from the center of all images -> shape (4, 3, 10, 10)
crop_patch = images[:, :, 9:19, 9:19]
print("10x10 cropped patch shape:", crop_patch.shape)

# Using Ellipsis (...) to grab the first pixel across all dimensions
print("Top-left pixel of all channels in all images:", images[..., 0, 0].shape)

print("-" * 40)

# 3. Boolean / Mask Indexing
mask = x > 50
print("Condition mask (x > 50):
", mask)
print("Filtered values greater than 50:", x[mask])


## 3. The Optimization Step (`torch.optim`)

Instead of updating parameters manually, PyTorch provides **optimizers** (`SGD`, `Adam`) to handle parameter updates.

### The Standard 5-Step Training Loop
1. `optimizer.zero_grad()` ? Clear previous gradients.
2. `preds = model(x)` ? Forward pass.
3. `loss = criterion(preds, targets)` ? Compute loss.
4. `loss.backward()` ? Backpropagate gradients.
5. `optimizer.step()` ? Update parameters ($	heta \leftarrow 	heta - \eta \cdot 
abla 	ext{Loss}$).


In [ ]:
import torch
import torch.optim as optim

# Synthetic dataset: y = 2*x + 1
X = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])

# Model parameters (random initial guesses)
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# Optimizer
optimizer = optim.SGD([w, b], lr=0.05)

# Train for 50 steps
for epoch in range(50):
    optimizer.zero_grad()
    pred = X * w + b
    loss = ((pred - y) ** 2).mean()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d} | Loss: {loss.item():.4f} | w: {w.item():.2f}, b: {b.item():.2f}")


> <small>**What does `torch.nn` do?**<br>
> ? **Auto-Tracks Parameters**: `model.parameters()` finds all weights and biases automatically?no manual tracking needed.<br>
> ? **Prebuilt Building Blocks**: Provides standard layers (`nn.Linear`, `nn.Conv2d`), activations, and loss functions.<br>
> ? **Device Control**: `model.to("cuda")` transfers the entire network and its weights to GPU in one command.<br>
> ? **Lifecycle Management**: Toggles behavior for training vs. inference via `model.train()` and `model.eval()`.</small>

## 4. Building Neural Networks with `nn.Module`

In PyTorch, custom neural networks subclass `nn.Module`:
- `__init__`: Define your layers and components here.
- `forward(x)`: Defines the execution flow of data through the network.
- **Never call `model.forward(x)` directly**?call `model(x)` so PyTorch can run internal hooks and autograd setups.


In [ ]:
import torch
import torch.nn as nn

# Define a 2-layer Neural Network
class SimpleMLP(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.layer1 = nn.Linear(in_features, hidden_features)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_features, out_features)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# Instantiate model
model = SimpleMLP(in_features=4, hidden_features=8, out_features=2)
print(model)

# Forward pass with a batch of 3 samples
dummy_input = torch.randn(3, 4)
output = model(dummy_input)
print("
Output shape:", output.shape)  # (3, 2)
